# Performance Evaluation and Optimization

In this final notebook, we'll evaluate the performance characteristics of structured generation and explore optimization techniques.

## Setup

In [ ]:
import outlines
import json
import time
import statistics
import matplotlib.pyplot as plt
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Literal, Optional
import psutil
import threading

In [ ]:
# Initialize model
model = outlines.models.transformers("microsoft/DialoGPT-medium")
print("Model loaded successfully!")

## 1. Comprehensive Performance Benchmark

Let's create a comprehensive benchmark comparing different generation approaches:

In [ ]:
# Define test schemas of varying complexity
class SimpleFinancialData(BaseModel):
    symbol: str
    price: float
    volume: int

class MediumFinancialData(BaseModel):
    symbol: str = Field(pattern=r"^[A-Z]{1,5}$")
    price: float = Field(ge=0)
    volume: int = Field(ge=0)
    sector: Literal["technology", "healthcare", "finance", "energy"]
    metrics: List[float] = Field(min_items=3, max_items=5)

class ComplexFinancialData(BaseModel):
    symbol: str = Field(pattern=r"^[A-Z]{1,5}$")
    company_name: str = Field(max_length=100)
    price: float = Field(ge=0, le=10000)
    volume: int = Field(ge=0)
    market_cap: float = Field(ge=0)
    sector: Literal["technology", "healthcare", "finance", "energy", "consumer_goods"]
    exchange: Literal["NYSE", "NASDAQ", "AMEX"]
    financial_metrics: dict = Field(description="Key financial ratios")
    analyst_ratings: List[str] = Field(min_items=1, max_items=10)
    last_updated: str = Field(pattern=r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")

# Performance testing framework
class PerformanceMonitor:
    def __init__(self):
        self.start_time = None
        self.end_time = None
        self.memory_usage = []
        self.monitoring = False
        
    def start_monitoring(self):
        self.start_time = time.time()
        self.memory_usage = []
        self.monitoring = True
        
        # Start memory monitoring in background
        def monitor_memory():
            while self.monitoring:
                self.memory_usage.append(psutil.Process().memory_info().rss / 1024 / 1024)  # MB
                time.sleep(0.1)
        
        self.monitor_thread = threading.Thread(target=monitor_memory)
        self.monitor_thread.start()
    
    def stop_monitoring(self):
        self.end_time = time.time()
        self.monitoring = False
        if hasattr(self, 'monitor_thread'):
            self.monitor_thread.join()
    
    def get_metrics(self):
        return {
            'duration': self.end_time - self.start_time if self.end_time else None,
            'avg_memory': statistics.mean(self.memory_usage) if self.memory_usage else 0,
            'peak_memory': max(self.memory_usage) if self.memory_usage else 0,
            'memory_samples': len(self.memory_usage)
        }

def benchmark_generation(schema_class, prompt, iterations=5):
    """Benchmark generation performance for a given schema"""
    generator = outlines.generate.json(model, schema_class)
    
    results = []
    
    for i in range(iterations):
        monitor = PerformanceMonitor()
        monitor.start_monitoring()
        
        try:
            result = generator(prompt)
            success = True
        except Exception as e:
            result = str(e)
            success = False
        
        monitor.stop_monitoring()
        metrics = monitor.get_metrics()
        
        results.append({
            'iteration': i + 1,
            'success': success,
            'duration': metrics['duration'],
            'avg_memory': metrics['avg_memory'],
            'peak_memory': metrics['peak_memory'],
            'output_size': len(json.dumps(result)) if success else 0
        })
    
    return results

print("Performance monitoring framework ready!")

## 2. Run Comprehensive Benchmarks

Let's benchmark all three complexity levels:

In [ ]:
# Define test prompts
prompts = {
    'simple': "Generate stock data for Apple Inc.",
    'medium': "Generate detailed stock information for a technology company with good performance metrics.",
    'complex': "Generate comprehensive financial data for Microsoft including all metrics, analyst ratings, and recent updates."
}

schemas = {
    'simple': SimpleFinancialData,
    'medium': MediumFinancialData, 
    'complex': ComplexFinancialData
}

# Run benchmarks
benchmark_results = {}

print("Running performance benchmarks...")
print("This may take a few minutes...")

for complexity, schema in schemas.items():
    print(f"\nBenchmarking {complexity} schema...")
    results = benchmark_generation(schema, prompts[complexity], iterations=3)
    benchmark_results[complexity] = results
    
    # Print summary
    successful_results = [r for r in results if r['success']]
    if successful_results:
        avg_duration = statistics.mean([r['duration'] for r in successful_results])
        avg_memory = statistics.mean([r['avg_memory'] for r in successful_results])
        print(f"  Average duration: {avg_duration:.2f}s")
        print(f"  Average memory: {avg_memory:.1f}MB")
        print(f"  Success rate: {len(successful_results)}/{len(results)}")
    else:
        print(f"  All attempts failed")

print("\nBenchmarking complete!")

## 3. Traditional vs Structured Generation Comparison

Let's compare structured generation with traditional free-form generation:

In [ ]:
# Traditional generation benchmark
def benchmark_traditional_generation(prompt, iterations=3):
    """Benchmark traditional text generation"""
    generator = outlines.generate.text(model)
    
    results = []
    
    for i in range(iterations):
        monitor = PerformanceMonitor()
        monitor.start_monitoring()
        
        result = generator(prompt, max_tokens=200)
        
        monitor.stop_monitoring()
        metrics = monitor.get_metrics()
        
        results.append({
            'iteration': i + 1,
            'duration': metrics['duration'],
            'avg_memory': metrics['avg_memory'],
            'peak_memory': metrics['peak_memory'],
            'output_length': len(result)
        })
    
    return results

# Compare traditional vs structured
comparison_prompt = "Generate financial information for Apple Inc. including stock price, volume, and company details."

print("Comparing Traditional vs Structured Generation:")
print("=" * 50)

# Traditional generation
traditional_results = benchmark_traditional_generation(comparison_prompt, 3)
traditional_avg_time = statistics.mean([r['duration'] for r in traditional_results])
traditional_avg_memory = statistics.mean([r['avg_memory'] for r in traditional_results])

print(f"Traditional Generation:")
print(f"  Average time: {traditional_avg_time:.2f}s")
print(f"  Average memory: {traditional_avg_memory:.1f}MB")
print(f"  Output format: Free text")

# Structured generation (medium complexity)
structured_results = benchmark_results['medium']
structured_successful = [r for r in structured_results if r['success']]
if structured_successful:
    structured_avg_time = statistics.mean([r['duration'] for r in structured_successful])
    structured_avg_memory = statistics.mean([r['avg_memory'] for r in structured_successful])
    
    print(f"\nStructured Generation (Medium):")
    print(f"  Average time: {structured_avg_time:.2f}s")
    print(f"  Average memory: {structured_avg_memory:.1f}MB")
    print(f"  Output format: Valid JSON")
    
    print(f"\nPerformance Comparison:")
    time_overhead = (structured_avg_time / traditional_avg_time - 1) * 100
    memory_overhead = (structured_avg_memory / traditional_avg_memory - 1) * 100
    print(f"  Time overhead: {time_overhead:+.1f}%")
    print(f"  Memory overhead: {memory_overhead:+.1f}%")
    print(f"  Benefit: Guaranteed valid, parseable output")

## 4. Optimization Techniques

Let's explore various optimization techniques for structured generation:

In [ ]:
# Optimization 1: Schema Simplification
class OptimizedFinancialData(BaseModel):
    # Fewer constraints, simpler patterns
    symbol: str = Field(max_length=5)
    price: float = Field(ge=0)
    volume: int = Field(ge=0)
    sector: Literal["tech", "health", "finance"]  # Fewer options
    # Removed complex nested structures

# Optimization 2: Caching generators
class GeneratorCache:
    def __init__(self):
        self.cache = {}
    
    def get_generator(self, schema_class):
        schema_name = schema_class.__name__
        if schema_name not in self.cache:
            print(f"Creating new generator for {schema_name}")
            self.cache[schema_name] = outlines.generate.json(model, schema_class)
        else:
            print(f"Using cached generator for {schema_name}")
        return self.cache[schema_name]

generator_cache = GeneratorCache()

# Test optimization impact
def test_optimization(schema_class, prompt, use_cache=False):
    """Test generation with different optimization strategies"""
    
    if use_cache:
        generator = generator_cache.get_generator(schema_class)
    else:
        generator = outlines.generate.json(model, schema_class)
    
    start_time = time.time()
    result = generator(prompt)
    end_time = time.time()
    
    return {
        'duration': end_time - start_time,
        'result': result
    }

# Compare optimization strategies
print("Testing Optimization Strategies:")
print("=" * 40)

test_prompt = "Generate stock data for Tesla."

# Test 1: Complex schema without cache
complex_no_cache = test_optimization(MediumFinancialData, test_prompt, use_cache=False)
print(f"Complex schema (no cache): {complex_no_cache['duration']:.2f}s")

# Test 2: Complex schema with cache (second call)
complex_with_cache = test_optimization(MediumFinancialData, test_prompt, use_cache=True)
print(f"Complex schema (cached): {complex_with_cache['duration']:.2f}s")

# Test 3: Optimized schema
optimized_result = test_optimization(OptimizedFinancialData, test_prompt, use_cache=False)
print(f"Optimized schema: {optimized_result['duration']:.2f}s")

print(f"\nOptimization Results:")
cache_improvement = (complex_no_cache['duration'] - complex_with_cache['duration']) / complex_no_cache['duration'] * 100
schema_improvement = (complex_no_cache['duration'] - optimized_result['duration']) / complex_no_cache['duration'] * 100

print(f"Caching improvement: {cache_improvement:.1f}%")
print(f"Schema simplification: {schema_improvement:.1f}%")

## 5. Quality vs Performance Trade-offs

Let's analyze the trade-offs between generation quality and performance:

In [ ]:
# Quality assessment function
def assess_output_quality(output, schema_class):
    """Assess the quality of generated output"""
    quality_metrics = {
        'valid_json': False,
        'schema_compliant': False,
        'field_completeness': 0,
        'data_realism': 0  # Subjective score 0-10
    }
    
    try:
        # Check if it's valid JSON
        if isinstance(output, dict):
            quality_metrics['valid_json'] = True
        
        # Check schema compliance
        validated = schema_class(**output)
        quality_metrics['schema_compliant'] = True
        
        # Field completeness (how many fields are non-empty)
        total_fields = len(output)
        non_empty_fields = sum(1 for v in output.values() if v is not None and v != "" and v != [])
        quality_metrics['field_completeness'] = non_empty_fields / total_fields if total_fields > 0 else 0
        
        # Basic realism check (simple heuristics)
        realism_score = 5  # Start with neutral
        
        # Check for realistic financial values
        if 'price' in output and isinstance(output['price'], (int, float)):
            if 0.01 <= output['price'] <= 10000:  # Reasonable stock price range
                realism_score += 1
        
        if 'volume' in output and isinstance(output['volume'], int):
            if output['volume'] > 0:  # Positive volume
                realism_score += 1
        
        if 'symbol' in output and isinstance(output['symbol'], str):
            if 1 <= len(output['symbol']) <= 5 and output['symbol'].isalpha():
                realism_score += 1
        
        quality_metrics['data_realism'] = min(realism_score, 10)
        
    except Exception as e:
        pass  # Keep default values
    
    return quality_metrics

# Compare quality across different schema complexities
schemas_for_quality = [
    (SimpleFinancialData, "simple"),
    (OptimizedFinancialData, "optimized"),
    (MediumFinancialData, "medium")
]

quality_results = {}

print("Quality vs Performance Analysis:")
print("=" * 40)

for schema_class, name in schemas_for_quality:
    # Generate sample and measure performance
    start_time = time.time()
    generator = outlines.generate.json(model, schema_class)
    output = generator("Generate financial data for a major technology stock.")
    generation_time = time.time() - start_time
    
    # Assess quality
    quality = assess_output_quality(output, schema_class)
    
    quality_results[name] = {
        'generation_time': generation_time,
        'quality': quality,
        'output': output
    }
    
    print(f"\n{name.capitalize()} Schema:")
    print(f"  Generation time: {generation_time:.2f}s")
    print(f"  Valid JSON: {quality['valid_json']}")
    print(f"  Schema compliant: {quality['schema_compliant']}")
    print(f"  Field completeness: {quality['field_completeness']:.1%}")
    print(f"  Data realism: {quality['data_realism']}/10")
    print(f"  Sample output: {json.dumps(output, indent=2)[:100]}...")

## 6. Performance Visualization

Let's create visualizations to better understand the performance characteristics:

In [ ]:
# Create performance comparison charts
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Chart 1: Generation Time by Complexity
complexities = list(quality_results.keys())
times = [quality_results[c]['generation_time'] for c in complexities]

ax1.bar(complexities, times, color=['lightblue', 'lightgreen', 'lightcoral'])
ax1.set_title('Generation Time by Schema Complexity')
ax1.set_ylabel('Time (seconds)')
ax1.set_xlabel('Schema Type')

# Chart 2: Quality Metrics Comparison
quality_metrics = ['valid_json', 'schema_compliant', 'field_completeness', 'data_realism']
x_pos = range(len(complexities))
width = 0.2

for i, metric in enumerate(quality_metrics[:3]):  # Skip data_realism for now
    values = []
    for complexity in complexities:
        val = quality_results[complexity]['quality'][metric]
        if isinstance(val, bool):
            val = 1.0 if val else 0.0
        values.append(val)
    
    ax2.bar([x + i * width for x in x_pos], values, width, 
            label=metric.replace('_', ' ').title())

ax2.set_title('Quality Metrics by Schema Type')
ax2.set_xlabel('Schema Type')
ax2.set_ylabel('Score')
ax2.set_xticks([x + width for x in x_pos])
ax2.set_xticklabels(complexities)
ax2.legend()
ax2.set_ylim(0, 1.1)

# Chart 3: Time vs Quality Trade-off
times_for_plot = [quality_results[c]['generation_time'] for c in complexities]
quality_scores = [quality_results[c]['quality']['field_completeness'] for c in complexities]

ax3.scatter(times_for_plot, quality_scores, s=100, c=['blue', 'green', 'red'])
for i, complexity in enumerate(complexities):
    ax3.annotate(complexity, (times_for_plot[i], quality_scores[i]), 
                xytext=(5, 5), textcoords='offset points')

ax3.set_title('Performance vs Quality Trade-off')
ax3.set_xlabel('Generation Time (seconds)')
ax3.set_ylabel('Field Completeness Score')
ax3.grid(True, alpha=0.3)

# Chart 4: Memory Usage Simulation
# Simulate memory usage patterns for different schema types
schema_types = ['Simple', 'Optimized', 'Medium', 'Complex']
base_memory = [100, 120, 180, 250]  # Simulated baseline memory usage
peak_memory = [120, 150, 220, 320]  # Simulated peak memory usage

x_pos = range(len(schema_types))
ax4.bar(x_pos, base_memory, width=0.4, label='Base Memory', alpha=0.7)
ax4.bar([x + 0.4 for x in x_pos], peak_memory, width=0.4, label='Peak Memory', alpha=0.7)

ax4.set_title('Memory Usage by Schema Complexity')
ax4.set_xlabel('Schema Type')
ax4.set_ylabel('Memory (MB)')
ax4.set_xticks([x + 0.2 for x in x_pos])
ax4.set_xticklabels(schema_types)
ax4.legend()

plt.tight_layout()
plt.show()

print("Performance analysis complete!")

## 7. Recommendations and Best Practices

Based on our performance analysis, let's provide actionable recommendations:

In [ ]:
# Generate performance recommendations
def generate_recommendations(quality_results):
    """Generate performance recommendations based on test results"""
    
    recommendations = {
        "General Performance": [],
        "Schema Design": [],
        "Production Deployment": [],
        "Quality Assurance": []
    }
    
    # Analyze results to generate specific recommendations
    fastest_schema = min(quality_results.keys(), 
                        key=lambda x: quality_results[x]['generation_time'])
    highest_quality = max(quality_results.keys(), 
                         key=lambda x: quality_results[x]['quality']['field_completeness'])
    
    # General Performance
    recommendations["General Performance"].extend([
        f"Fastest generation: {fastest_schema} schema ({quality_results[fastest_schema]['generation_time']:.2f}s)",
        "Use generator caching for repeated operations",
        "Consider async generation for multiple requests",
        "Monitor memory usage in production environments"
    ])
    
    # Schema Design
    recommendations["Schema Design"].extend([
        "Start with simple schemas and add complexity gradually",
        "Use specific constraints only when necessary",
        "Prefer enums over complex regex patterns for better performance",
        "Limit nested structures depth to maintain speed"
    ])
    
    # Production Deployment
    recommendations["Production Deployment"].extend([
        "Implement request timeouts for structured generation",
        "Use load balancing for high-throughput scenarios",
        "Cache frequently used generators",
        "Monitor generation success rates and adjust schemas accordingly"
    ])
    
    # Quality Assurance
    recommendations["Quality Assurance"].extend([
        f"Best quality schema: {highest_quality}",
        "Implement automated validation checks",
        "Regular testing with diverse prompts",
        "Monitor output consistency across generations"
    ])
    
    return recommendations

# Create performance summary
def create_performance_summary(quality_results):
    """Create a comprehensive performance summary"""
    
    summary = {
        "total_tests": len(quality_results),
        "avg_generation_time": statistics.mean([r['generation_time'] for r in quality_results.values()]),
        "success_rate": sum(1 for r in quality_results.values() 
                           if r['quality']['schema_compliant']) / len(quality_results),
        "best_performance": min(quality_results.keys(), 
                               key=lambda x: quality_results[x]['generation_time']),
        "best_quality": max(quality_results.keys(), 
                           key=lambda x: quality_results[x]['quality']['field_completeness'])
    }
    
    return summary

# Generate and display recommendations
recommendations = generate_recommendations(quality_results)
summary = create_performance_summary(quality_results)

print("PERFORMANCE ANALYSIS SUMMARY")
print("=" * 50)
print(f"Total schemas tested: {summary['total_tests']}")
print(f"Average generation time: {summary['avg_generation_time']:.2f}s")
print(f"Overall success rate: {summary['success_rate']:.1%}")
print(f"Best performance: {summary['best_performance']} schema")
print(f"Best quality: {summary['best_quality']} schema")

print("\nRECOMMENDATIONS")
print("=" * 50)

for category, recs in recommendations.items():
    print(f"\n{category}:")
    for rec in recs:
        print(f"  • {rec}")

# Performance optimization checklist
print("\nPERFORMANCE OPTIMIZATION CHECKLIST")
print("=" * 50)
checklist = [
    "☐ Cache generators for repeated use",
    "☐ Use simple schemas when possible", 
    "☐ Implement timeout mechanisms",
    "☐ Monitor memory usage",
    "☐ Test with production-like data",
    "☐ Validate outputs automatically",
    "☐ Plan for error handling",
    "☐ Consider async processing for scale"
]

for item in checklist:
    print(item)

## 8. Production Readiness Assessment

Let's create a final assessment for production readiness:

In [ ]:
# Production readiness checklist
def assess_production_readiness(quality_results):
    """Assess if the structured generation setup is production-ready"""
    
    criteria = {
        "Performance": {
            "avg_time_under_5s": statistics.mean([r['generation_time'] for r in quality_results.values()]) < 5.0,
            "consistent_performance": True,  # Simplified for demo
            "memory_efficient": True  # Simplified for demo
        },
        "Quality": {
            "high_success_rate": sum(1 for r in quality_results.values() 
                                   if r['quality']['schema_compliant']) / len(quality_results) >= 0.95,
            "complete_outputs": all(r['quality']['field_completeness'] > 0.8 
                                  for r in quality_results.values()),
            "realistic_data": all(r['quality']['data_realism'] >= 6 
                                for r in quality_results.values())
        },
        "Reliability": {
            "error_handling": True,  # Simplified for demo
            "graceful_degradation": True,  # Simplified for demo
            "monitoring_ready": True  # Simplified for demo
        }
    }
    
    # Calculate overall readiness score
    total_checks = sum(len(checks) for checks in criteria.values())
    passed_checks = sum(sum(checks.values()) for checks in criteria.values())
    readiness_score = passed_checks / total_checks
    
    return criteria, readiness_score

# Assess production readiness
criteria, readiness_score = assess_production_readiness(quality_results)

print("PRODUCTION READINESS ASSESSMENT")
print("=" * 50)
print(f"Overall Readiness Score: {readiness_score:.1%}")
print()

for category, checks in criteria.items():
    print(f"{category}:")
    for check, passed in checks.items():
        status = "✅ PASS" if passed else "❌ FAIL"
        check_name = check.replace('_', ' ').title()
        print(f"  {check_name}: {status}")
    print()

# Readiness recommendation
if readiness_score >= 0.9:
    recommendation = "READY FOR PRODUCTION"
    color = "🟢"
elif readiness_score >= 0.7:
    recommendation = "READY WITH MONITORING"
    color = "🟡"
else:
    recommendation = "NEEDS IMPROVEMENT"
    color = "🔴"

print(f"{color} RECOMMENDATION: {recommendation}")
print()

# Final notes
print("FINAL NOTES:")
print("• Structured generation provides significant benefits for financial applications")
print("• Performance overhead is acceptable for most use cases")
print("• Quality and consistency improvements justify the additional complexity")
print("• Consider your specific use case requirements when choosing schema complexity")
print("• Always implement proper monitoring and error handling in production")

## Summary

In this comprehensive evaluation, we've explored:

1. **Performance Characteristics**: Different schema complexities have predictable performance impacts
2. **Quality Trade-offs**: More complex schemas generally provide better structure but at performance cost
3. **Optimization Strategies**: Caching, schema simplification, and proper design can significantly improve performance
4. **Production Considerations**: Structured generation is viable for production with proper monitoring and optimization

**Key Takeaways for Financial Applications:**
- Structured generation solves real problems in financial data processing
- The performance overhead is justified by the reliability and consistency benefits
- Start simple and add complexity only when needed
- Always implement proper testing and monitoring

This concludes our deep dive into structured generation with the Outlines library!